In [1]:
# 1. Import required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load pharmacy sales dataset
df = pd.read_csv("global_pharmacy_sales_2020_2025_daily_dataset.csv")

print("Dataset loaded successfully! Total records:", len(df))
df.head()

Dataset loaded successfully! Total records: 177990


,date,year,month,day,region,country,category,medicine,age_group,units_sold,unit_price,stock_level,expiry_days_remaining,covid_flag
0,2025-10-12,2025,10,12,Middle East,UAE,Chronic,Amlodipine,0-12,328,45.38,4774,466,0
1,2020-09-15,2020,9,15,South Asia,India,Chronic,Amlodipine,26-45,371,75.49,4584,181,1
2,2020-02-26,2020,2,26,Middle East,UAE,Vitamin,Vitamin C,0-12,948,22.51,3934,556,1
3,2025-11-09,2025,11,9,South Asia,Sri Lanka,Chronic,Amlodipine,0-12,275,63.29,4544,330,0
4,2022-04-04,2022,4,4,South America,Argentina,Chronic,Amlodipine,26-45,563,44.28,4284,590,0


In [2]:
# 2. Select useful columns for pharmacy analysis
cols = [
    'region', 'country', 'category', 'medicine', 'age_group',
    'units_sold', 'unit_price', 'stock_level',
    'expiry_days_remaining', 'covid_flag'
]
df = df[cols].copy()

# Convert numerical columns
num_cols = ['units_sold', 'unit_price', 'stock_level', 'expiry_days_remaining']
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop missing or invalid records
df = df.dropna().copy()
df = df[df['units_sold'] >= 0].copy()

# Create target variable: High vs Low sales status based on median
df['sales_status'] = np.where(
    df['units_sold'] >= df['units_sold'].median(),
    'High', 'Low'
)

print("Sales classes distribution:")
print(df['sales_status'].value_counts())
df.head()

Sales classes distribution:
sales_status
High    89217
Low     88773
Name: count, dtype: int64


,region,country,category,medicine,age_group,units_sold,unit_price,stock_level,expiry_days_remaining,covid_flag,sales_status
0,Middle East,UAE,Chronic,Amlodipine,0-12,328,45.38,4774,466,0,Low
1,South Asia,India,Chronic,Amlodipine,26-45,371,75.49,4584,181,1,Low
2,Middle East,UAE,Vitamin,Vitamin C,0-12,948,22.51,3934,556,1,High
3,South Asia,Sri Lanka,Chronic,Amlodipine,0-12,275,63.29,4544,330,0,Low
4,South America,Argentina,Chronic,Amlodipine,26-45,563,44.28,4284,590,0,High


In [3]:
# 3. Select features (X) and target (y)
X = df.drop(['units_sold', 'sales_status'], axis=1)
y = df['sales_status']

# Numerical continuous features for binning
num_cols = ['unit_price', 'stock_level', 'expiry_days_remaining']

# Split data into 80% train and 20% test (stratified by target)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Create copies for baseline and binned experiments
X_base_train = X_train.copy()
X_base_test = X_test.copy()

X_bin_train = X_train.copy()
X_bin_test = X_test.copy()

print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")

Train samples: 142392, Test samples: 35598


In [4]:
# 4. Train baseline model before binning (using raw continuous values)
# Convert categorical features into dummy variables
X_base_train = pd.get_dummies(X_base_train, dtype=int)
X_base_test = pd.get_dummies(X_base_test, dtype=int)

# Align columns between train and test sets
X_base_test = X_base_test.reindex(columns=X_base_train.columns, fill_value=0)

# Train baseline decision tree
base_model = DecisionTreeClassifier(max_depth=5, random_state=42)
base_model.fit(X_base_train, y_train)

# Predict sales status
base_pred = base_model.predict(X_base_test)

# Display baseline accuracy
print("Accuracy before binning:", accuracy_score(y_test, base_pred))

Accuracy before binning: 0.8201865273329962


In [5]:
# 5. Apply Decision Tree-Based Feature Binning
# This finds optimal cut points for continuous features using the target variable
for col in num_cols:
    # Shallow decision tree to determine split thresholds
    tree = DecisionTreeClassifier(
        max_leaf_nodes=4,
        min_samples_leaf=0.05,
        random_state=42
    )
    # Fit tree on single numerical feature
    tree.fit(X_bin_train[[col]], y_train)

    # Extract cut points from tree nodes
    limits = sorted(set(tree.tree_.threshold[tree.tree_.feature >= 0]))

    # Define bin boundaries and labels
    bins = [-np.inf] + limits + [np.inf]
    labels = [f'Group {i+1}' for i in range(len(bins) - 1)]

    # Transform both train and test data with the same bins
    X_bin_train[col] = pd.cut(X_bin_train[col], bins=bins, labels=labels).astype(str)
    X_bin_test[col] = pd.cut(X_bin_test[col], bins=bins, labels=labels).astype(str)

print("Feature binning completed!")
X_bin_train.head()

Feature binning completed!


,region,country,category,medicine,age_group,unit_price,stock_level,expiry_days_remaining,covid_flag
142468,South America,Brazil,Antibiotic,Azithromycin,13-25,Group 4,Group 3,Group 2,0
150210,South Asia,Bangladesh,Antibiotic,Azithromycin,65+,Group 2,Group 4,Group 2,0
97406,South America,Argentina,Cough_Cold,Cetirizine,65+,Group 4,Group 1,Group 2,0
6395,Africa,Kenya,Chronic,Amlodipine,26-45,Group 4,Group 1,Group 3,1
6592,Africa,Kenya,Antipyretic,Ibuprofen,26-45,Group 2,Group 3,Group 4,0


In [6]:
# 6. Display learned group ranges for each feature (Interpretability)
for col in num_cols:
    tree = DecisionTreeClassifier(
        max_leaf_nodes=4,
        min_samples_leaf=0.05,
        random_state=42
    )
    tree.fit(X_train[[col]], y_train)

    limits = sorted(set(tree.tree_.threshold[tree.tree_.feature >= 0]))
    bins = [-np.inf] + limits + [np.inf]

    print(f"\nFeature: {col}")
    for i in range(len(bins) - 1):
        print(f"  Group {i+1}: {bins[i]:.2f} to {bins[i+1]:.2f}")


Feature: unit_price
  Group 1: -inf to 11.55
  Group 2: 11.55 to 30.95
  Group 3: 30.95 to 59.98
  Group 4: 59.98 to inf



Feature: stock_level
  Group 1: -inf to 4448.50
  Group 2: 4448.50 to 4530.50
  Group 3: 4530.50 to 4726.50
  Group 4: 4726.50 to inf



Feature: expiry_days_remaining
  Group 1: -inf to 107.50
  Group 2: 107.50 to 419.50
  Group 3: 419.50 to 594.50
  Group 4: 594.50 to inf


In [7]:
# 7. Train model on binned data
# Convert categorical and newly binned features into dummy variables
X_bin_train = pd.get_dummies(X_bin_train, dtype=int)
X_bin_test = pd.get_dummies(X_bin_test, dtype=int)

# Align columns between train and test sets
X_bin_test = X_bin_test.reindex(columns=X_bin_train.columns, fill_value=0)

# Train decision tree on binned data
bin_model = DecisionTreeClassifier(max_depth=5, random_state=42)
bin_model.fit(X_bin_train, y_train)

# Predict sales status
bin_pred = bin_model.predict(X_bin_test)

# Display accuracy after binning
print("Accuracy after binning:", accuracy_score(y_test, bin_pred))

Accuracy after binning: 0.8201865273329962


In [8]:
# 8. Compare model performance before and after binning
acc_before = accuracy_score(y_test, base_pred)
acc_after = accuracy_score(y_test, bin_pred)

print("Accuracy before binning:", round(acc_before, 4))
print("Accuracy after binning: ", round(acc_after, 4))

if acc_after > acc_before:
    print("Result: Binning improved classification accuracy!")
elif acc_after < acc_before:
    print("Result: Binning reduced classification accuracy.")
else:
    print("Result: Both models achieve the exact same accuracy with higher interpretability!")

# Detailed classification reports
print("\n--- Classification Report (Before Binning) ---")
print(classification_report(y_test, base_pred, zero_division=0))

print("\n--- Classification Report (After Binning) ---")
print(classification_report(y_test, bin_pred, zero_division=0))

Accuracy before binning: 0.8202
Accuracy after binning:  0.8202
Result: Both models achieve the exact same accuracy with higher interpretability!

--- Classification Report (Before Binning) ---


              precision    recall  f1-score   support

        High       0.85      0.78      0.81     17843
         Low       0.80      0.86      0.83     17755

    accuracy                           0.82     35598
   macro avg       0.82      0.82      0.82     35598
weighted avg       0.82      0.82      0.82     35598


--- Classification Report (After Binning) ---


              precision    recall  f1-score   support

        High       0.85      0.78      0.81     17843
         Low       0.80      0.86      0.83     17755

    accuracy                           0.82     35598
   macro avg       0.82      0.82      0.82     35598
weighted avg       0.82      0.82      0.82     35598

